### Silver Layer transforamtion

###Data loading

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Connection with Data Lake

In [0]:
storage_account = "adventurework"
secrets = dbutils.secrets.get(scope = 'adventurescope',key = 'app-secrets')
application_id = "8acfcd9f-4e48-4f99-a6e5-4fe6a467caaa"
directory_id = "3211f43b-ee38-4d64-95d0-a29cad376485"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", application_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",secrets)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{directory_id}/oauth2/token")

### Creating DataFrame for Each Dataset

In [0]:
basepath = "abfss://adventurework@adventurework.dfs.core.windows.net/bronze/"
path_calender = basepath + "AdventureWorks_Calendar.csv"
path_customers = basepath + "AdventureWorks_Customers.csv"
path_product_cat = basepath + "AdventureWorks_Product_Categories.csv"
path_product_subcat = basepath + "AdventureWorks_Product_Subcategories.csv"
path_product = basepath + "AdventureWorks_Products.csv"
path_return = basepath + "AdventureWorks_Returns.csv"
path_sales2015 = basepath + "AdventureWorks_Sales_2015.csv"
path_sales2016 = basepath + "AdventureWorks_Sales_2016.csv"
path_sales2017 = basepath + "AdventureWorks_Sales_2017.csv"
path_territory = basepath + "AdventureWorks_Territories.csv"



In [0]:
df_calender = spark.read.format("csv").option("header",True).option("inferSchema",True).load(path_calender)
df_customer = spark.read.format("csv").option("header",True).option("inferSchema",True).load(path_customers)
df_product_cat = spark.read.format("csv").option("header",True).option("inferSchema",True).load(path_product_cat)
df_product_subcat = spark.read.format("csv").option("header",True).option("inferSchema",True).load(path_product_subcat)
df_product = spark.read.format("csv").option("header",True).option("inferSchema",True).load(path_product)
df_return = spark.read.format("csv").option("header",True).option("inferSchema",True).load(path_return)
df_sales2015 = spark.read.format("csv").option("header",True).option("inferSchema",True).load(path_sales2015)
df_sales2016 = spark.read.format("csv").option("header",True).option("inferSchema",True).load(path_sales2016)
df_sales2017 = spark.read.format("csv").option("header",True).option("inferSchema",True).load(path_sales2017)
df_territory = spark.read.format("csv").option("header",True).option("inferSchema",True).load(path_territory)


In [0]:
df_calender = df_calender.withColumn("month",month(col("Date")))\
               .withColumn("year",year(col("Date")))

### Sales Data 2015

In [0]:
df_customer_sales2015 = df_sales2015.join(df_customer,df_sales2015.CustomerKey == df_customer.CustomerKey,"left")
df_cust_sales2015_prod = df_customer_sales2015.join(df_product,df_customer_sales2015.ProductKey ==df_product.ProductKey,"left")
df_cust_sales2015_prod = df_cust_sales2015_prod.withColumn("CustomerName",expr("Prefix || ' ' || FirstName || ' ' || LastName"))
df_cust_sales2015_prod_sub = df_product_subcat.join(df_cust_sales2015_prod,df_cust_sales2015_prod.ProductSubcategoryKey == df_product_subcat.ProductSubcategoryKey,"left")
df_cust_sales2015_prod_sub_date = df_calender.join(df_cust_sales2015_prod_sub,df_cust_sales2015_prod_sub.OrderDate == df_calender.Date,"left")
df_cust_sales2015_prod_sub_date_ter = df_territory.join(df_cust_sales2015_prod_sub_date,df_cust_sales2015_prod_sub_date.TerritoryKey == df_territory.SalesTerritoryKey,"left")
df_cust_sales2015_prod_sub_date_ter = df_cust_sales2015_prod_sub_date_ter.select(df_customer.CustomerKey,"CustomerName","Orderdate",col("month").alias("Order Month"),col("year").alias("Order Year"),"StockDate","OrderNumber","OrderLineItem",df_product.ProductKey,"TerritoryKey",df_territory.Region,df_territory.Country,df_territory.Continent,"OrderQuantity",df_product.ProductName,df_product.ProductCost,"ModelName",df_product.ProductPrice,"ProductColor","ProductStyle","ProductSize","ProductDescription",df_product_subcat.ProductSubcategoryKey,df_product_subcat.SubcategoryName,"AnnualIncome","EmailAddress")

### Sales data 2016

In [0]:
df_customer_sales2016 = df_sales2016.join(df_customer,df_sales2016.CustomerKey == df_customer.CustomerKey,"left")
df_cust_sales2016_prod = df_customer_sales2016.join(df_product,df_customer_sales2016.ProductKey ==df_product.ProductKey,"left")
df_cust_sales2016_prod = df_cust_sales2016_prod.withColumn("CustomerName",expr("Prefix || ' ' || FirstName || ' ' || LastName"))
df_cust_sales2016_prod_sub = df_product_subcat.join(df_cust_sales2016_prod,df_cust_sales2016_prod.ProductSubcategoryKey == df_product_subcat.ProductSubcategoryKey,"left")
df_cust_sales2016_prod_sub_date = df_calender.join(df_cust_sales2016_prod_sub,df_cust_sales2016_prod_sub.OrderDate == df_calender.Date,"left")
df_cust_sales2016_prod_sub_date_ter = df_territory.join(df_cust_sales2016_prod_sub_date,df_cust_sales2016_prod_sub_date.TerritoryKey == df_territory.SalesTerritoryKey,"left")
df_cust_sales2016_prod_sub_date_ter = df_cust_sales2016_prod_sub_date_ter.select(df_customer.CustomerKey,"CustomerName","Orderdate",col("month").alias("Order Month"),col("year").alias("Order Year"),"StockDate","OrderNumber","OrderLineItem",df_product.ProductKey,"TerritoryKey",df_territory.Region,df_territory.Country,df_territory.Continent,"OrderQuantity",df_product.ProductName,df_product.ProductCost,"ModelName",df_product.ProductPrice,"ProductColor","ProductStyle","ProductSize","ProductDescription",df_product_subcat.ProductSubcategoryKey,df_product_subcat.SubcategoryName,"AnnualIncome","EmailAddress")
display(df_cust_sales2016_prod_sub_date_ter.head(5))

CustomerKey,CustomerName,Orderdate,Order Month,Order Year,StockDate,OrderNumber,OrderLineItem,ProductKey,TerritoryKey,Region,Country,Continent,OrderQuantity,ProductName,ProductCost,ModelName,ProductPrice,ProductColor,ProductStyle,ProductSize,ProductDescription,ProductSubcategoryKey,SubcategoryName,AnnualIncome,EmailAddress
21452,MR. SETH BAILEY,2016-12-31,12,2016,2003-11-27,SO61128,3,214,1,Northwest,United States,North America,1,"Sport-100 Helmet, Red",13.0863,Sport-100,34.99,Red,0,0,"Universal fit, well-vented, lightweight , snap-on visor.",31,Helmets,"$40,000",seth87@adventure-works.com
21452,MR. SETH BAILEY,2016-12-31,12,2016,2003-11-06,SO61128,1,478,1,Northwest,United States,North America,2,Mountain Bottle Cage,3.7363,Mountain Bottle Cage,9.99,NA,0,0,Tough aluminum cage holds bottle securly on tough terrain.,28,Bottles and Cages,"$40,000",seth87@adventure-works.com
21452,MR. SETH BAILEY,2016-12-31,12,2016,2003-12-08,SO61128,2,477,1,Northwest,United States,North America,2,Water Bottle - 30 oz.,1.8663,Water Bottle,4.99,NA,0,0,AWC logo water bottle - holds 30 oz; leak-proof.,28,Bottles and Cages,"$40,000",seth87@adventure-works.com
19654,MR. RICHARD ALEXANDER,2016-12-31,12,2016,2003-10-10,SO61129,1,481,1,Northwest,United States,North America,1,"Racing Socks, M",3.3623,Racing Socks,8.99,White,U,M,"Thin, lightweight and durable with cuffs that stay up.",23,Socks,"$40,000",richard74@adventure-works.com
19654,MR. RICHARD ALEXANDER,2016-12-31,12,2016,2003-12-03,SO61129,2,475,1,Northwest,United States,North America,1,"Women's Mountain Shorts, M",26.1763,Women's Mountain Shorts,69.99,Black,W,M,"Heavy duty, abrasion-resistant shorts feature seamless, lycra inner shorts with anti-bacterial chamois for comfort.",22,Shorts,"$40,000",richard74@adventure-works.com


### Sales data 2017

In [0]:
df_customer_sales2017 = df_sales2017.join(df_customer,df_sales2017.CustomerKey == df_customer.CustomerKey,"left")
df_cust_sales2017_prod = df_customer_sales2017.join(df_product,df_customer_sales2017.ProductKey ==df_product.ProductKey,"left")
df_cust_sales2017_prod = df_cust_sales2017_prod.withColumn("CustomerName",expr("Prefix || ' ' || FirstName || ' ' || LastName"))
df_cust_sales2017_prod_sub = df_product_subcat.join(df_cust_sales2017_prod,df_cust_sales2017_prod.ProductSubcategoryKey == df_product_subcat.ProductSubcategoryKey,"left")
df_cust_sales2017_prod_sub_date = df_calender.join(df_cust_sales2017_prod_sub,df_cust_sales2017_prod_sub.OrderDate == df_calender.Date,"left")
df_cust_sales2017_prod_sub_date_ter = df_territory.join(df_cust_sales2017_prod_sub_date,df_cust_sales2017_prod_sub_date.TerritoryKey == df_territory.SalesTerritoryKey,"left")
df_cust_sales2017_prod_sub_date_ter = df_cust_sales2017_prod_sub_date_ter.select(df_customer.CustomerKey,"CustomerName","Orderdate",col("month").alias("Order Month"),col("year").alias("Order Year"),"StockDate","OrderNumber","OrderLineItem",df_product.ProductKey,"TerritoryKey",df_territory.Region,df_territory.Country,df_territory.Continent,"OrderQuantity",df_product.ProductName,df_product.ProductCost,"ModelName",df_product.ProductPrice,"ProductColor","ProductStyle","ProductSize","ProductDescription",df_product_subcat.ProductSubcategoryKey,df_product_subcat.SubcategoryName,"AnnualIncome","EmailAddress")
display(df_cust_sales2017_prod_sub_date_ter.head(5))

CustomerKey,CustomerName,Orderdate,Order Month,Order Year,StockDate,OrderNumber,OrderLineItem,ProductKey,TerritoryKey,Region,Country,Continent,OrderQuantity,ProductName,ProductCost,ModelName,ProductPrice,ProductColor,ProductStyle,ProductSize,ProductDescription,ProductSubcategoryKey,SubcategoryName,AnnualIncome,EmailAddress
22253,MRS. STEPHANIE REED,2017-06-30,6,2017,2004-03-18,SO74142,2,490,1,Northwest,United States,North America,1,"Short-Sleeve Classic Jersey, L",41.5723,Short-Sleeve Classic Jersey,53.99,Yellow,U,L,"Short sleeve classic breathable jersey with superior moisture control, front zipper, and 3 back pockets.",21,Jerseys,"$60,000",stephanie7@adventure-works.com
22253,MRS. STEPHANIE REED,2017-06-30,6,2017,2004-05-30,SO74142,1,383,1,Northwest,United States,North America,1,"Road-550-W Yellow, 40",605.6492,Road-550-W,1000.4375,Yellow,W,40,"Same technology as all of our Road series bikes, but the frame is sized for a woman. Perfect all-around bike for road or racing.",2,Road Bikes,"$60,000",stephanie7@adventure-works.com
11717,MR. MARCUS JONES,2017-06-30,6,2017,2004-06-07,SO74125,2,537,1,Northwest,United States,North America,1,HL Mountain Tire,13.09,HL Mountain Tire,35.0,NA,0,0,"Incredible traction, lightweight carbon reinforced.",37,Tires and Tubes,"$60,000",marcus3@adventure-works.com
11717,MR. MARCUS JONES,2017-06-30,6,2017,2004-05-06,SO74125,1,528,1,Northwest,United States,North America,1,Mountain Tire Tube,1.8663,Mountain Tire Tube,4.99,NA,0,0,Self-sealing tube.,37,Tires and Tubes,"$60,000",marcus3@adventure-works.com
11717,MR. MARCUS JONES,2017-06-30,6,2017,2004-03-26,SO74125,3,471,1,Northwest,United States,North America,1,"Classic Vest, S",23.749,Classic Vest,63.5,Blue,U,S,"Light-weight, wind-resistant, packs to fit into a pocket.",25,Vests,"$60,000",marcus3@adventure-works.com


In [0]:
combined_sales_df = df_cust_sales2015_prod_sub_date_ter.union(df_cust_sales2016_prod_sub_date_ter).union(df_cust_sales2017_prod_sub_date_ter)
display(combined_sales_df.head(5))

CustomerKey,CustomerName,Orderdate,Order Month,Order Year,StockDate,OrderNumber,OrderLineItem,ProductKey,TerritoryKey,Region,Country,Continent,OrderQuantity,ProductName,ProductCost,ModelName,ProductPrice,ProductColor,ProductStyle,ProductSize,ProductDescription,ProductSubcategoryKey,SubcategoryName,AnnualIncome,EmailAddress
26624,MS. ELIZABETH RUSSELL,2015-12-28,12,2015,2002-09-15,SO48695,1,362,1,Northwest,United States,North America,1,"Mountain-200 Black, 46",1105.81,Mountain-200,2049.0982,Black,U,46,Serious back-country riding. Perfect for all levels of competition. Uses the same HL Frame as the Mountain-100.,1,Mountain Bikes,"$50,000",elizabeth48@adventure-works.com
13729,MR. JASON LI,2015-12-26,12,2015,2002-09-16,SO48676,1,370,1,Northwest,United States,North America,1,"Road-250 Red, 52",1518.7864,Road-250,2443.35,Red,U,52,"Alluminum-alloy frame provides a light, stiff ride, whether you are racing in the velodrome or on a demanding club ride on country roads.",2,Road Bikes,"$40,000",jason24@adventure-works.com
14283,MR. SETH SIMMONS,2015-12-23,12,2015,2002-10-29,SO48645,1,387,1,Northwest,United States,North America,1,"Road-550-W Yellow, 44",605.6492,Road-550-W,1000.4375,Yellow,W,44,"Same technology as all of our Road series bikes, but the frame is sized for a woman. Perfect all-around bike for road or racing.",2,Road Bikes,"$60,000",seth63@adventure-works.com
13722,MR. LUKE CAMPBELL,2015-12-23,12,2015,2002-10-15,SO48646,1,370,1,Northwest,United States,North America,1,"Road-250 Red, 52",1518.7864,Road-250,2443.35,Red,U,52,"Alluminum-alloy frame provides a light, stiff ride, whether you are racing in the velodrome or on a demanding club ride on country roads.",2,Road Bikes,"$30,000",luke38@adventure-works.com
26625,MS. LAUREN BARNES,2015-12-23,12,2015,2002-10-23,SO48647,1,354,1,Northwest,United States,North America,1,"Mountain-200 Silver, 42",1117.8559,Mountain-200,2071.4196,Silver,U,42,Serious back-country riding. Perfect for all levels of competition. Uses the same HL Frame as the Mountain-100.,1,Mountain Bikes,"$60,000",lauren50@adventure-works.com


### loading Sales Data in Azure Datalake (Silver Layer)

In [0]:
combined_sales_df.write.mode("overwrite").format("parquet").save("abfss://adventurework@adventurework.dfs.core.windows.net/silver/")

In [0]:
dbutils.fs.ls(dbutils.widgets.get("dbfss_path"))

---------------------------------------------------------------------------
ExecutionError                            Traceback (most recent call last)
File <command-4645096743256543>, line 1
----> 1 dbutils.fs.ls(dbutils.widgets.get("dbfss_path"))

File /databricks/python_shell/lib/dbruntime/remotefshandler/RemoteFsHandler.py:52, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
     49 class ExecutionError(Exception):
     50     pass
---> 52 raise ExecutionError(str(e)) from None

ExecutionError: (java.io.FileNotFoundException) Operation failed: "The specified path does not exist.", 404, GET, https://adventurework.dfs.core.windows.net/adventurework?upn=false&resource=filesystem&maxResults=5000&directory=deltadest/customers&timeout=90&recursive=false, PathNotFound, , "The specified path does not exist. RequestId:9e5c0933-901f-0010-1f85-a87e7c000000 Time:2026-02-28T07:39:10.7191730Z"

JVM stacktrace:
java.io.FileNotFoundException
	at shaded.databricks.a

### dbutils.secrets

In [0]:
dbutils.secrets.list(scope = 'adventurescope')

[SecretMetadata(key='app-secrets')]

In [0]:
dbutils.secrets.get(scope = 'adventurescope',key = 'app-secrets')

'[REDACTED]'

In [0]:
dbutils.widgets.text("dbfss_path",'abfss://adventurework@adventurework.dfs.core.windows.net/silver')

In [0]:
dbutils.widgets.get("dbfss_path")

In [0]:
dbutils.fs.mkdirs(dbutils.widgets.get("dbfss_path"))

True

# Delta Lake

In [0]:
emp_df = spark.read.format("csv")\
                   .option("header",True)\
                       .option("inferSchema",True)\
                       .load('abfss://adventurework@adventurework.dfs.core.windows.net/deltasource/')


emp_df.write.format("delta")\
     .option("mode","overwrite")\
    .save(dbutils.widgets.get("dbfss_path"))


In [0]:
dbutils.fs.ls(dbutils.widgets.get("dbfss_path"))

## managed delta tables

In [0]:
%sql
create database salesDB

In [0]:
%sql
create table salesDB.customers
(
  id int,
  name string,
  number int
) using DELTA

In [0]:
%sql
select * from salesdb.customers;

id,name,number
1,aa,34
2,bb,45
3,cc,32
4,dd,57


### insert data into managed delta table

In [0]:
%sql
insert into salesdb.customers
values
(1,"aa",34),
(2,"bb",45),
(3,"cc",32),
(4,"dd",57)

num_affected_rows,num_inserted_rows
4,4


In [0]:
%sql
drop table salesdb.customers


### external delta table

In [0]:
%sql
create table salesDB.customers
(
  id int,
  name string,
  number int
) using DELTA
location 'abfss://adventurework@adventurework.dfs.core.windows.net/sales_data/'

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4593485765790212>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "create table salesDB.customers\n(\n  id int,\n  name string,\n  number int\n) using DELTA\nlocation 'abfss://adventurework@adventurework.dfs.core.windows.net/sales_data/'\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_